# 10 · Front-end impairments

Between the DSP and the channel sit analog components: the IQ modulator,
DACs, drivers, the 90° hybrid, photodiodes and ADCs.  Their imperfections
are deterministic and mostly simple.  This notebook covers the classic
ones, each with how a receiver removes it:

1. **IQ imbalance**: unequal gains and a non-90° angle between I and Q,
   and a time skew between them;
2. **DC offsets**;
3. **limited electrical bandwidth**, which causes ISI;
4. **carrier frequency offset** between the transmitter laser and the LO;
5. **the ADC**: quantization and sampling jitter.

The chain where they all appear together is
[24 · End-to-end coherent transceiver](24-coherent-transceiver-end-to-end.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.impairments import GramSchmidt, Lowdin
from commkit.recovery import ConstantPhase

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
WIDE = np.multiply(plt.rcParams["figure.figsize"], (2, 1))


def host(a):
    return to_device(a, "cpu")


def snr_db(y):
    """SNR after removing a static rotation against the known symbols.

    Metrics never rotate on their own; ConstantPhase() fits one common phase.
    """
    return ck.metrics.snr(ck.recovery.correct_carrier_phase(y, ConstantPhase()))

## 1. IQ imbalance

An ideal receiver measures `I = Re{r}` and `Q = Im{r}` with equal gains on
orthogonal axes.  A real one has an amplitude imbalance `ε` (in dB) and a
phase error `φ`, which mixes in the **mirror image** of the signal:

$$r = \mu\, s + \nu\, s^*,$$

with an image-rejection ratio `|μ|²/|ν|²` of about 25-30 dB for typical
devices (the example below exaggerates it).  The square grid of a QAM constellation becomes a skewed
parallelogram.

In [ ]:
tx = ck.generate(C16, num_symbols=2**15, symbol_rate=32e9, rng=1, device=DEVICE)
rx = ck.impairments.apply_awgn(tx, esn0_db=25, rng=2)
iq = ck.impairments.apply_iq_imbalance(
    rx, amplitude_imbalance_db=1.5, phase_imbalance_deg=8.0
)

fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(rx, ax=axes[0], title="Balanced receiver")
ck.plotting.plot_constellation(iq, ax=axes[1], title="1.5 dB, 8° IQ imbalance")
plt.show()

# The image term: a single tone at +f acquires a mirror at -f.
fs = 1e6
t = np.arange(2**14) / fs
tone = ck.backend.to_device(np.exp(2j * np.pi * 100e3 * t).astype(np.complex64), DEVICE)
tone_iq = ck.impairments.apply_iq_imbalance(
    tone, amplitude_imbalance_db=1.5, phase_imbalance_deg=8.0
)
spec = np.abs(np.fft.fft(host(tone_iq))) ** 2
k = int(round(100e3 / fs * t.size))
print(f"image rejection ratio: {10 * np.log10(spec[k] / spec[-k]):.1f} dB")
_, ax = ck.plotting.plot_psd(
    tone, sampling_rate=fs, nperseg=1024, title=None, label="balanced"
)
ck.plotting.plot_psd(
    tone_iq,
    sampling_rate=fs,
    nperseg=1024,
    ax=ax,
    title="A 100 kHz tone and its mirror image at -100 kHz",
    label="1.5 dB, 8° imbalance",
    legend=True,
    show=True,
)

### Blind correction

Because a QAM (or any circular) signal has I and Q of equal power and
uncorrelated, the imbalance can be removed blindly by re-orthogonalizing
the two branches:

* `GramSchmidt()` keeps I as the reference, removes its projection from Q
  and normalizes both (the classic GSOP);
* `Lowdin()` treats both branches symmetrically (whitening of the 2x2
  covariance), spreading the correction evenly.

Both restore the SNR.  They differ by a small common rotation (GSOP keeps
the I axis, Löwdin splits the difference), which carrier recovery removes
anyway; the SNRs below are measured after a data-aided derotation.

The constellations below show the impaired symbols and the Gram-Schmidt
output: the parallelogram becomes a square grid again, slightly rotated
(the rotation is the carrier's business, not the imbalance's).

In [ ]:
print(f"SNR balanced         {snr_db(rx):5.2f} dB")
print(f"SNR with imbalance   {snr_db(iq):5.2f} dB")
for how in (GramSchmidt(), Lowdin()):
    fixed = ck.impairments.correct_iq_imbalance(iq, how)
    print(f"SNR after {type(how).__name__:12s} {snr_db(fixed):5.2f} dB")

fixed = ck.impairments.correct_iq_imbalance(iq, GramSchmidt())
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(iq, ax=axes[0], title="Before IQ correction")
ck.plotting.plot_constellation(fixed, ax=axes[1], title="After Gram-Schmidt")
plt.show()

### Time skew

The I and Q paths of a receiver (the hybrid's outputs, cables, the ADC
channels) never have exactly the same length.  A **skew** `τ` delays Q
against I.  A delay is a phase shift that grows with frequency, `2π f τ`, so
unlike the gain and phase errors above, the skew's error depends on
frequency: a tone at `f` acquires a mirror image of relative power
`tan²(π f τ)`.  Near DC it does nothing; the band edges suffer most.
`apply_iq_imbalance` takes it as `skew=` in seconds, with the sampling rate
from the Signal.

Below, 4 ps of skew (an eighth of a 32 GBd symbol, about 0.8 mm of extra
fibre or 0.6 mm of circuit-board trace): first the image rejection of single
tones across the band, measured from the spectrum, against `1/tan²(π f τ)`.

In [ ]:
SKEW = 4e-12  # 4 ps
fs = 64e9
n = 2**12
t = np.arange(n) / fs
tone_f = np.array([1, 2, 4, 6, 8, 12, 16, 20, 24, 28]) * 1e9
irr = []
for f in tone_f:
    f = round(f / fs * n) * fs / n  # on an FFT bin
    tone = to_device(np.exp(2j * np.pi * f * t).astype(np.complex64), DEVICE)
    r = ck.impairments.apply_iq_imbalance(
        tone,
        amplitude_imbalance_db=0.0,
        phase_imbalance_deg=0.0,
        skew=SKEW,
        sampling_rate=fs,
    )
    spec = np.abs(np.fft.fft(host(r))) ** 2
    k = round(f / fs * n)
    irr.append(10 * np.log10(spec[k] / spec[-k]))
f_fine = np.linspace(0.5e9, 30e9, 200)
fig, ax = plt.subplots()
ax.plot(f_fine / 1e9, -20 * np.log10(np.tan(np.pi * f_fine * SKEW)), color="0.6",
        label="1 / tan²(π f τ)")  # fmt: skip
ax.plot(tone_f / 1e9, irr, "o", label="measured, single tones")
ax.axhline(25, color="C3", linestyle=":", label="typical static imbalance")
ax.axvspan(0, 17.6, color="C0", alpha=0.08, label="16-QAM band, RRC 0.1")
ax.set_xlabel("Tone frequency [GHz]")
ax.set_ylabel("Image rejection ratio [dB]")
ax.set_title("4 ps of IQ skew")
ax.legend()
plt.show()

At 2 GHz the image is 32 dB down, at the band edge of a 32 GBd signal
(17.6 GHz with a 0.1 rolloff) only 13 dB: worse than a typical static
imbalance over most of the band.

On 16-QAM at 2 samples per symbol the skew costs 9 dB.  Besides the image,
Q is sampled an eighth of a symbol away from the centre of its eye while I
is sampled at the centre: the Q coordinate of every point is smeared by its
neighbours, and the constellation spreads vertically.  The blind
corrections are memoryless: they see the covariance of I and Q at one
instant and cannot undo a delay, so Gram-Schmidt leaves the SNR where the
skew put it.  Removing skew takes a filter per branch: a deskew filter
(a fractional delay of I by `τ`, if `τ` is known from calibration) or a
widely linear equalizer, which learns it.

In [ ]:
qam = ck.impairments.apply_awgn(
    ck.generate(
        C16,
        num_symbols=2**15,
        symbol_rate=32e9,
        sps=2,
        pulse=RRC(rolloff=0.1),
        rng=3,
        device=DEVICE,
    ),
    esn0_db=25,
    rng=4,
)
skewed = ck.impairments.apply_iq_imbalance(
    qam, amplitude_imbalance_db=0.0, phase_imbalance_deg=0.0, skew=SKEW
)
deskewed = ck.impairments.correct_iq_imbalance(skewed, GramSchmidt())


def receive(x):
    return ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))


fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2.4, 0.8))
)
for ax, (name, x) in zip(axes, (
    ("no skew", qam), ("4 ps skew", skewed), ("skew, then Gram-Schmidt", deskewed)
), strict=True):  # fmt: skip
    y = receive(x)
    print(f"{name:25s} SNR {snr_db(y):5.2f} dB")
    ck.plotting.plot_constellation(y, ax=ax, title=name)
plt.show()

## 2. DC offsets

Imperfectly balanced photodiode pairs and ADC offsets add a constant to I
and Q: a spectral line at 0 Hz and a shifted constellation.  It is removed
by subtracting the mean, *before* the IQ-imbalance correction (which assumes
a zero-mean signal).  In self-homodyne measurements the line sits in the
middle of the signal and needs a dark capture instead (see
[28](28-laser-linewidth-self-homodyne-iq.ipynb)).

In [ ]:
dc = 0.15 - 0.1j
off = iq.replace(samples=iq.samples + dc)
restored = ck.impairments.correct_iq_imbalance(
    off.replace(samples=off.samples - off.samples.mean()), GramSchmidt()
)
print(f"estimated DC {complex(off.samples.mean()):.3f} (true {dc})")
print(
    f"SNR: with DC and imbalance {snr_db(off):.2f} dB, corrected {snr_db(restored):.2f} dB"
)
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(off, ax=axes[0], title="DC offset and IQ imbalance")
ck.plotting.plot_constellation(restored, ax=axes[1], title="Mean removed, then GSOP")
plt.show()

## 3. Limited electrical bandwidth

DACs, drivers and photodiodes low-pass filter the waveform.  A bandwidth
approaching half the symbol rate widens each pulse into its neighbours,
and the eye closes: ISI.  Modeled here as a 4th-order Bessel response
(03 · section 3) with its -3 dB point at `cutoff` (`norm="mag"`), on a
32 GBd RRC signal at 2 samples per symbol.  The adaptive
equalizer removes this ISI later
([17 · Adaptive equalization](17-adaptive-equalization.ipynb)).

In [ ]:
tx2 = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=3,
    device=DEVICE,
)


def bandwidth_limit(sig, cutoff):
    """A causal 4th-order Bessel front end.  Its bulk (DC) group delay,
    which timing recovery removes in a receiver, is taken out here."""
    sos = ck.filtering.bessel_sos(
        sampling_rate=sig.sampling_rate, cutoff=cutoff, order=4, norm="mag"
    )
    b, a = sos[:, :3], sos[:, 3:]
    k = np.arange(3)
    delay = np.sum(b @ k / b.sum(axis=1) - a @ k / a.sum(axis=1))  # samples
    filtered = ck.filtering.iir_filter(sig, sos)
    return ck.timing.apply_delay(filtered, delay=-delay)


fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 0.75))
)
_, ax_psd = plt.subplots()
for ax, bw in zip(axes, (None, 20e9, 12e9), strict=True):
    x = tx2 if bw is None else bandwidth_limit(tx2, bw)
    y = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    label = "no bandwidth limit" if bw is None else f"Bessel {bw / 1e9:.0f} GHz"
    print(f"{label:20s} signal-to-ISI {ck.metrics.snr(y):5.1f} dB")
    ck.plotting.plot_constellation(y, ax=ax, title=label)
    ck.plotting.plot_psd(
        x,
        nperseg=1024,
        ax=ax_psd,
        title="The front end attenuates the band edges",
        label=label,
        legend=True,
    )
plt.show()

## 4. Carrier frequency offset

The transmitter laser and the LO are independent, and their frequencies
differ by up to a few GHz.  The received baseband is shifted by that
offset `Δf`: the constellation **spins** at `Δf` turns per second, which
over one symbol is `2π Δf T` radians.  Even 100 MHz at 32 GBd is a full
turn every 320 symbols, far faster than phase noise; frequency-offset
estimation removes it first
([13 · Frequency-offset estimation](13-frequency-offset-estimation.ipynb)).

In [ ]:
spun = ck.spectral.shift_frequency(tx, frequency=100e6)
phase = np.unwrap(np.angle(host(spun.samples) * np.conj(host(tx.samples))))
slope = np.polyfit(np.arange(phase.size), phase, 1)[0]
print(
    f"phase advance per symbol {slope:.5f} rad, 2π Δf T = {2 * np.pi * 100e6 / 32e9:.5f} rad"
)

fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(
    spun.samples[:100], ax=axes[0], title="First 100 symbols", constellation=C16
)
ck.plotting.plot_constellation(spun, ax=axes[1], title="Whole record: a ring")
plt.show()

## 5. The ADC: quantization and sampling jitter

### Quantization

An ADC with `b` bits rounds each component to one of `2^b` levels between
`±full_scale` (`apply_quantization`); anything beyond the full scale is
**clipped** to the outermost level.  The step is `Δ = 2·full_scale/2^b`, and
the rounding error, spread evenly over `±Δ/2`, has the variance `Δ²/12`.
Below, a stretch of the in-phase component of 16-QAM at 2 samples per
symbol and its 3-bit version for two full scales: twice the component's
rms, where the peaks are clipped, and four times, where nothing clips but
the steps are twice as coarse.

In [ ]:
rms = float(np.sqrt(np.mean(host(tx2.samples).real ** 2)))  # one component


def sliced_snr(x):
    return ck.metrics.snr(
        ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    )


segment = slice(2 * 300, 2 * 340)
i_wave = host(tx2.samples.real[segment]) / rms
fig, axes = plt.subplots(1, 2, figsize=WIDE, sharey=True)
for ax, load in zip(axes, (2.0, 4.0), strict=True):
    adc = ck.impairments.apply_quantization(tx2, num_bits=3, full_scale=load * rms)
    ax.plot(i_wave, ".-", color="0.6", label="input")
    ax.step(np.arange(i_wave.size), host(adc.samples.real[segment]) / rms, where="mid",
            color="C0", label="3-bit ADC")  # fmt: skip
    for edge in (-load, load):
        ax.axhline(edge, color="C3", linestyle=":")
    ax.set_title(f"full scale = {load:.0f} × rms")
    ax.set_xlabel("Sample")
axes[0].set_ylabel("In-phase amplitude / rms")
axes[0].legend(loc="lower left")
plt.show()

A full-scale sine uses every level evenly and gives the classic
`6.02·b + 1.76` dB.  A modulated signal is not a sine: its large values are
rare, so the full scale is set a few times its rms, the **loading**.  Too low a full scale clips the peaks, too
high a one wastes levels on values that almost never occur.  The histogram
of the component shows why: its values cluster around the four levels of
16-QAM, with a spread of values in between from the pulses' tails, and only
1.4 % of them lie beyond twice the rms, practically none beyond three
times.

In [ ]:
comp = host(tx2.samples.real) / rms
fig, ax = plt.subplots()
ax.hist(comp, bins=200, density=True, color="C0")
for load, color in ((2, "C1"), (3, "C2"), (4, "C3")):
    beyond = np.mean(np.abs(comp) > load)
    ax.axvline(
        load,
        color=color,
        linestyle=":",
        label=f"{load} × rms: {100 * beyond:.2f} % beyond",
    )
    ax.axvline(-load, color=color, linestyle=":")
ax.set_xlabel("In-phase amplitude / rms")
ax.set_ylabel("Probability density")
ax.legend()
plt.show()

Without clipping, the SNR after the matched filter follows from the
rounding noise alone: `Δ²/12` per component against the component's power
`σ²`, plus 3 dB because the matched filter keeps the signal and only the
half of the noise inside the signal's band at 2 samples per symbol:

$$\mathrm{SNR} = 6.02\,b + 4.77 + 3.01 - 20\log_{10} L \ \mathrm{dB},$$

for the loading `L = full_scale / σ`.  Below, the measured SNR for 3 to 10
bits and three loadings against these lines.  The record itself stops at
44.5 dB without quantization (the pulse's truncation).

In [ ]:
bits = np.arange(3, 11)
fig, ax = plt.subplots()
print("bits   2 x rms   3 x rms   4 x rms   [SNR in dB]")
table = []
for load, color in ((2.0, "C1"), (3.0, "C2"), (4.0, "C3")):
    snr = [
        sliced_snr(
            ck.impairments.apply_quantization(
                tx2, num_bits=int(b), full_scale=load * rms
            )
        )
        for b in bits
    ]
    table.append(snr)
    ax.plot(bits, snr, "o-", color=color, label=f"full scale {load:.0f} × rms")
    ax.plot(
        bits, 6.02 * bits + 7.78 - 20 * np.log10(load), "--", color=color, alpha=0.6
    )
for b, row in zip(bits, np.transpose(table), strict=True):
    print(f"{b:4d}  " + "  ".join(f"{s:8.2f}" for s in row))
ax.axhline(sliced_snr(tx2), color="0.6", linestyle=":", label="no quantization")
ax.set_xlabel("ADC resolution [bits]")
ax.set_ylabel("SNR after the matched filter [dB]")
ax.set_title("16-QAM, 2 samples per symbol (dashed: rounding only)")
ax.legend()
plt.show()

At low resolution the rounding dominates, the loadings follow their lines
(except at 3 bits with the largest full scale, where a step is as large as
the rms and the rounding error is no longer spread evenly), and the
smallest full scale wins: twice the rms for 3 to 5 bits.  With more
bits the rounding noise falls by 6 dB per bit but the clipping does not: the
curve for twice the rms flattens at the clipping's own distortion, and the
best loading rises to three times the rms from 6 bits on.

### Sampling jitter

The sampling instants of a real ADC also jitter around the clock's grid
(`apply_sampling_jitter`, rms `σ` in seconds).  Sampling a signal slightly
early or late gives an error proportional to its slope, and a tone at `f`
has a slope proportional to `f`: the jitter noise has relative power
`(2π f σ)²`.  Datasheets express an ADC's quality as the **effective number
of bits**, `ENOB = (SNR - 1.76)/6.02`, the resolution of an ideal converter
with the same SNR on a full-scale sine; with jitter it falls with the input
frequency.  The `enob=` argument of `apply_quantization` adds the
converter's other noise, lumped, on top of the rounding.

Below, a full-scale tone through an 8-bit ADC with 100 fs and 300 fs of rms
jitter, against the sum of the rounding noise and `(2π f σ)²`.

In [ ]:
fs = 64e9
t = np.arange(2**14) / fs
tones = np.array([1.03, 2.07, 4.07, 8.1, 16.1, 24.3, 30.3]) * 1e9
f_fine = np.geomspace(0.8e9, 32e9, 100)
q_noise = (
    (2 / 2**8) ** 2 / 12 * 2 / 0.99**2
)  # rounding noise of I and Q against the tone
fig, ax = plt.subplots()
for jitter, color in ((100e-15, "C0"), (300e-15, "C1")):
    snr = []
    for f in tones:
        tone = to_device(
            (0.99 * np.exp(2j * np.pi * f * t)).astype(np.complex64), DEVICE
        )
        jittered = ck.impairments.apply_sampling_jitter(
            tone, jitter=jitter, sampling_rate=fs, rng=1
        )
        adc = ck.impairments.apply_quantization(jittered, num_bits=8, full_scale=1.0)
        snr.append(10 * np.log10(
            np.mean(np.abs(host(tone)) ** 2) / np.mean(np.abs(host(adc - tone)) ** 2)
        ))  # fmt: skip
    theory = -10 * np.log10(q_noise + (2 * np.pi * f_fine * jitter) ** 2)
    ax.semilogx(f_fine, theory, color=color, alpha=0.6)
    ax.semilogx(
        tones, snr, "o", color=color, label=f"{jitter * 1e15:.0f} fs rms jitter"
    )
    print(f"{jitter * 1e15:.0f} fs: ENOB " + ", ".join(
        f"{(s - 1.76) / 6.02:.2f} at {f / 1e9:.0f} GHz" for s, f in zip(snr, tones, strict=True)
        if f in (tones[0], tones[4], tones[-1])
    ))  # fmt: skip
enob = ax.secondary_yaxis(
    "right", functions=(lambda s: (s - 1.76) / 6.02, lambda e: 6.02 * e + 1.76)
)
enob.set_ylabel("ENOB [bits]")
ax.set_xlabel("Tone frequency [Hz]")
ax.set_ylabel("SNR [dB]")
ax.set_title("8-bit ADC, full-scale tone (lines: rounding + jitter)")
ax.legend()
plt.show()

At low frequencies the rounding of the 8 bits sets the SNR; above a few GHz
the jitter takes over and the SNR falls by 20 dB per decade.  (At 30 GHz,
close to the Nyquist frequency of 32 GHz, the measurement is a little
better than the formula.)  At 30 GHz, 100 fs of jitter costs two bits of
resolution and 300 fs almost four: high-speed ADCs need very clean clocks.

**Summary**

| impairment | signature | removed by |
|---|---|---|
| IQ imbalance | image at `-f`, skewed grid | `correct_iq_imbalance(GramSchmidt() or Lowdin())` |
| IQ skew | image growing with `f` | a deskew filter or widely linear equalizer |
| DC offset | line at 0 Hz, shifted grid | mean subtraction (or a dark capture) |
| bandwidth limit | ISI, closed eye | adaptive equalizer |
| frequency offset | spinning constellation | `correct_frequency_offset` |
| quantization | noise of `Δ²/12`, clipped peaks | resolution and loading |
| sampling jitter | noise growing with `f` | a cleaner clock |

**Next:** [11 · Frame synchronization and timing](11-frame-synchronization-and-timing.ipynb).